In [1]:
using Random, Distributions, Statistics, Printf, DelimitedFiles, Dates
using LinearAlgebra
using StatsBase
using QuantileRegressions
using Plots
const bb = 120 
const aa = 40
const N  = 2_701_767
const I0 = 3
const S0 = 2_701_767 - 3 
const n_iter = 1_000_000
include("functions.jl")
Random.seed!(2025)


Istar_obs = [
2, 6, 11, 14, 17, 23, 31, 38, 43, 46, 74, 91, 119, 138, 193, 255, 257,
324, 372, 412, 422, 407, 411, 450, 408, 394, 371, 416, 425, 388, 387,
369, 386, 365, 328, 314, 335, 298, 323, 300, 280, 285, 273, 254, 253,
211, 209, 232, 203, 217, 199, 206, 217, 182, 173, 176, 154, 166, 157
]
tau = length(Istar_obs)

model_tag_sym = :reciprocal

KMAX_UPPER = 30  


# Fixed output dir
out_dir = "output"
isdir(out_dir) || mkpath(out_dir)

header_cont = []
if model_tag_sym === :memoryless
    global header_cont = ["beta", "alpha", "gamma"]
elseif model_tag_sym === :powerlaw
    global header_cont = ["beta", "alpha", "gamma", "lambda_P"]
elseif model_tag_sym === :exponential
    global header_cont = ["beta", "alpha", "gamma", "lambda_E"]
elseif model_tag_sym === :reciprocal
    global header_cont = ["beta", "alpha", "gamma", "lambda_R"] 
elseif model_tag_sym === :sliding
    global header_cont = ["beta", "alpha", "gamma"]            
else
    error("Unknown model tag: $(model_tag_sym)")
end

c = 1

@info "[$(String(model_tag_sym))_model] Fitting chain $(c) (tau=$tau)"

Random.seed!(2025 + c)
initθ_chain = initθ_for_chain(model_tag_sym) 
t0 = Dates.now()

try
    samples, loglik_aug_vecs =
        mcmc_one_chain_with_Rstar!(Istar_obs, N,S0, I0;
            fit_mech=model_tag_sym,
            n_iter=n_iter,
            initθ=initθ_chain,
            KMAX_UPPER=KMAX_UPPER)

    if size(samples, 1) != n_iter
        error("Chain $c did not complete all iterations.")
    end

    # Save samples
    samples_filename = "samples_chain_$(c).csv"
    write_csv(joinpath(out_dir, samples_filename), header_cont, samples)

    # Save per-time log-likelihoods (thinned & post-burnin inside mcmc)
    loglik_filename = "loglik_chain_$(c).csv"
    write_csv(joinpath(out_dir, loglik_filename), ["loglik"], hcat(loglik_aug_vecs))
    el = Dates.value(Dates.now() - t0) / 1000
catch err
    el = Dates.value(Dates.now() - t0) / 1000
end

@info "Chain completed -> output dir: $out_dir"


[ Info: [reciprocal_model] Fitting chain 1 (tau=59)
[ Info: [reciprocal] iter 1000/1000000 elapsed=5.4s, rate=0.077, mean=[0.874, 0.00096, 0.420, 0.330], std=[0.1096, 0.000253, 0.0380, 0.0327] [ADAPT]
[ Info: [reciprocal] iter 2000/1000000 elapsed=9.9s, rate=0.080, mean=[0.783, 0.00103, 0.424, 0.351], std=[0.1130, 0.000201, 0.0277, 0.0306] [ADAPT]
[ Info: [reciprocal] iter 3000/1000000 elapsed=13.6s, rate=0.076, mean=[0.752, 0.00106, 0.427, 0.359], std=[0.1006, 0.000179, 0.0233, 0.0277] [ADAPT]
[ Info: [reciprocal] iter 4000/1000000 elapsed=17.4s, rate=0.072, mean=[0.736, 0.00107, 0.427, 0.365], std=[0.0910, 0.000164, 0.0218, 0.0261] [ADAPT]
[ Info: [reciprocal] iter 5000/1000000 elapsed=21.1s, rate=0.073, mean=[0.723, 0.00108, 0.421, 0.360], std=[0.0847, 0.000159, 0.0226, 0.0265] [ADAPT]
[ Info: [reciprocal] iter 6000/1000000 elapsed=24.8s, rate=0.074, mean=[0.712, 0.00114, 0.419, 0.341], std=[0.0810, 0.000198, 0.0214, 0.0464] [ADAPT]
[ Info: [reciprocal] iter 7000/1000000 elapsed=28.